In [154]:
from matplotlib import pyplot as plt

In [155]:
def show_image(title, img):
    plt.figure(figsize=(8, 6))
    plt.imshow(img, cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()

In [156]:
import cv2
import numpy as np
import re
from passporteye import read_mrz
import tempfile


def display_image(title, image):
    cv2.imshow(title, image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def deskew_image(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, 100, minLineLength=100, maxLineGap=10)

    if lines is not None:
        angles = [np.arctan2(y2 - y1, x2 - x1) for [[x1, y1, x2, y2]] in lines]
        median_angle = np.median(angles)
        (h, w) = image.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, np.degrees(median_angle), 1.0)
        image = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

    return image

def crop_pass(image):
    img = image.copy()
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    t = 255
    ID = []
    while t >= 0:
        temp_thresh = cv2.threshold(gray, t, 255, cv2.THRESH_BINARY)[1]
        temp_dilate = cv2.dilate(temp_thresh, cv2.getStructuringElement(cv2.MORPH_RECT, (6, 1)), iterations=5)
        cnts = cv2.findContours(temp_dilate, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)[0]
        flag = False
        for c in cnts:
            area = cv2.contourArea(c)
            if area > 0.35 * img.shape[0] * img.shape[1]:
                ID.append(c)
                flag = True
        if flag:
            break
        t -= 10
    
    if not ID:
        print("No suitable contour found, returning original image.")
        return img
    
    (x, y, w, h) = cv2.boundingRect(ID[0])
    cropped_img = img[y:y+h, x:x+w]
    #display_image("Cropped Passport", cropped_img)
    return cropped_img

def extract_mrz(image):
    """Extracts MRZ data from a given cropped passport image."""
    
    # ✅ Save image as a temporary file for passporteye
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as temp_file:
        temp_filename = temp_file.name
        cv2.imwrite(temp_filename, image)  # Write OpenCV image to file

    # ✅ Now, pass the file path to passporteye
    mrz = read_mrz(temp_filename, save_roi=True)

    if mrz:
        return mrz.to_dict()
    
    return None
def format_date(mrz_date):
    if len(mrz_date) == 6:
        return f"{mrz_date[4:6]}/{mrz_date[2:4]}/{mrz_date[0:2]}"
    return mrz_date

def clean_mrz_text(text):
    text = text.replace("<", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text

def format_mrz_data(mrz_data):
    first_name = clean_mrz_text(mrz_data.get("names", ""))
    surname = clean_mrz_text(mrz_data.get("surname", ""))

    first_name = re.sub(r'K+\s*K+', '', first_name).strip()
    surname = re.sub(r'K+\s*K+', '', surname).strip()

    formatted_data = {
        "First Name": first_name,  
        "Surname": surname,  
        "Passport Number": mrz_data.get("number", "").replace("<", ""),
        "Date of Birth": format_date(mrz_data.get("date_of_birth", "")),
        "Gender": "Male" if mrz_data.get("sex", "") == "M" else "Female",
        "Nationality": mrz_data.get("nationality", "").replace("<", ""),
        "Expiration Date": format_date(mrz_data.get("expiration_date", "")),
        "MRZ Raw Text": mrz_data.get("raw_text", ""),
    }
    return formatted_data

def process_passport(image_path):
    image = cv2.imread(image_path)

    if image is None:
        print(f"Error: Unable to load image. Check file path: {image_path}")
        return

    image = deskew_image(image)
    #display_image("Deskewed Image", image)

    cropped_image = crop_pass(image)  # Make sure this function returns a valid image

    if cropped_image is None:
        print("Error: Cropping failed, using original image instead.")
        cropped_image = image  

    #display_image("Cropped Image", cropped_image)

    mrz_data = extract_mrz(cropped_image)  # ✅ Now correctly passing a file path

    if mrz_data:
        formatted_mrz = format_mrz_data(mrz_data)

        print("\n--- Extracted Passport Details ---")
        for key, value in formatted_mrz.items():
            if key != "MRZ Raw Text":  
                print(f"{key}: {value}")

        print("\n--- MRZ Raw Text ---")
        print(mrz_data["raw_text"])
    else:
        print("\nNo MRZ Data Found.")

if __name__ == "__main__":
    image_path = "passport/10.PNG"  
    process_passport(image_path)



c:\Users\tarun.pithani\AppData\Local\Programs\Python\Python313\Lib\site-packages\passporteye\mrz\image.py:37: FutureWarning: The plugin infrastructure in `skimage.io` and the parameter `plugin` are deprecated since version 0.25 and will be removed in 0.27 (or later). To avoid this warning, please do not use the parameter `plugin`. Instead, use `imageio` or other I/O packages directly. See also `imread`.
  img = skimage_io.imread(file, as_gray=self.as_gray, plugin='imageio')
c:\Users\tarun.pithani\AppData\Local\Programs\Python\Python313\Lib\site-packages\passporteye\mrz\image.py:89: FutureWarning: `square` is deprecated since version 0.25 and will be removed in version 0.27. Use `skimage.morphology.footprint_rectangle` instead.
  m = morphology.square(self.square_size)



--- Extracted Passport Details ---
First Name: AHMED ALI AHMED SHAYAC
Surname: SHAYA
Passport Number: RRCF93040
Date of Birth: 13/03/98
Gender: Male
Nationality: ARE
Expiration Date: 31/07/27

--- MRZ Raw Text ---
P<ARESHAYA<<AHMED<ALI<AHMED<SHAYACKK<KKKKKKK
RRCF930409ARE9B03134M2707314<<<<0
